In [1]:
!pip install langchain_community langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import os
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_community.utilities import WikipediaAPIWrapper
from google.colab import userdata

# API Key
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API")

# LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3
)


In [21]:
@tool
def get_internal_stock_price(ticker: str) -> str:
    """Primary internal database for retrieving stock price.
    This database is currently failing."""
    return "Error: Database Timeout"


@tool
def search_public_web(query: str) -> str:
    """Backup method to search the public web for stock price if internal DB fails."""
    return "Apple stock is at $170"

In [22]:
tools = [get_internal_stock_price, search_public_web]

In [26]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a financial assistant.

Follow this reasoning process strictly:

1. First try the internal database using get_internal_stock_price.
2. Wait for the observation.
3. If the observation contains an error or timeout, then use search_public_web.
4. Finally answer the user.

Do not call both tools at the same time.
"""
)

In [27]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the current stock price of Apple?"}
    ]
})

print(response)

{'messages': [HumanMessage(content='What is the current stock price of Apple?', additional_kwargs={}, response_metadata={}, id='2b98d076-a69d-460f-8999-812a6becd42d'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '13w3xnrts', 'function': {'arguments': '{"ticker":"AAPL"}', 'name': 'get_internal_stock_price'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 377, 'total_tokens': 431, 'completion_time': 0.113985477, 'completion_tokens_details': None, 'prompt_time': 0.029101602, 'prompt_tokens_details': None, 'queue_time': 0.046020481, 'total_time': 0.143087079}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc38e-9f05-7f31-b665-db0075142b0d-0', tool_calls=[{'name': 'get_internal_stock_price', 'args': {'ticker': 'AAPL'}, 'id': '13w3xnrts', 'type': 'tool_call'}], invalid_too

In [28]:
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the current stock price of Apple?"}
        ]
    },
    debug=True
)

print(response)

[values] {'messages': [HumanMessage(content='What is the current stock price of Apple?', additional_kwargs={}, response_metadata={}, id='d4d2b64c-164c-41f9-8a8c-5f0059bffa86')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6tp0jyfrt', 'function': {'arguments': '{"ticker":"AAPL"}', 'name': 'get_internal_stock_price'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 377, 'total_tokens': 427, 'completion_time': 0.1087385, 'completion_tokens_details': None, 'prompt_time': 0.022346135, 'prompt_tokens_details': None, 'queue_time': 0.045874677, 'total_time': 0.131084635}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc38e-bf3c-7131-987f-64cb446068ad-0', tool_calls=[{'name': 'get_internal_stock_price', 'args': {'ticker': 'AAPL'}, 'id': '6tp0

In [29]:
messages = response["messages"]

reasoning_steps = []
final_answer = None

for msg in messages:

    # Tool calls (agent deciding to use tool)
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for call in msg.tool_calls:
            reasoning_steps.append(f"Action: {call['name']} | Input: {call['args']}")

    # Tool outputs
    if msg.__class__.__name__ == "ToolMessage":
        reasoning_steps.append(f"Observation: {msg.content}")

    # Final answer
    if msg.__class__.__name__ == "AIMessage" and msg.content:
        final_answer = msg.content

print("---- Agent Reasoning ----")
for step in reasoning_steps:
    print(step)

print("\n---- Final Answer ----")
print(final_answer)

---- Agent Reasoning ----
Action: get_internal_stock_price | Input: {'ticker': 'AAPL'}
Observation: Error: Database Timeout
Action: search_public_web | Input: {'query': 'AAPL current stock price'}
Observation: Apple stock is at $170

---- Final Answer ----
The current stock price of Apple is $170.
